### Paths, files & name outputs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

path = '\\' # contains data
path2 = '\\' # path to save processed data
file = ".txt" # data file
cell_number = 154 # for calibration parameters, if they are all in one file

lines = np.loadtxt(path+file)

nom_figure = ".".join(file.split('.')[:-1]) # to name the files

#fig, ax = plt.subplots(figsize=(100,20))
#plt.plot(lines[:,11], lines[:,0])
#plt.plot(lines[:,11], lines[:,3])
#plt.xticks(fontsize=30)
#plt.yticks(fontsize=30)

### Parameters

In [ ]:
# to be modified

distance_init = 3.00 *10**-6 # initial distance between particles in m

parameters = pd.read_csv('.csv', decimal=',',  sep=';', encoding='ISO-8859-1') # file containing calibration parameters
par = parameters.iloc[cell_number-1]
S1 = par[2] * 10**-3 / 10**-9
S2 = par[4] * 10**-3 / 10**-9
k1 = par[3] * 10**-12 / 10**-9
k2 = par[5] * 10**-12 / 10**-9
# if no file change par[] for measured value

keys = [1,2,3,4,5,6,7,8,9,10] # one key for each cycle
cycle = 20480 + 10240 + 20480 + 20480 # number of measurements per cycle (=time*frequency)
cycles = [cycle -1, 2*cycle -1, 3*cycle -1, 4*cycle -1, 5*cycle -1, 6*cycle -1, 7*cycle -1, 8*cycle -1, 9*cycle -1]
cuts = [20480 -1, 20480 +10240 -1, 20480 +10240 +20480 -1] # to cut inside cycles to separate each step
tot = 20480 + 10240 + 20480 -1

### Data retrieval and Force-Time plotting

In [ ]:
data_split = {}

start = 0
cut=tot

for i, end in enumerate(cycles):
    data_split[keys[i]] = lines[start:end]
    start=end
    
data_split[keys[-1]] = lines[start:]

dict = {}
coefs = []

fig1, ax = plt.subplots(figsize=(100,9))

for key in keys:
    dict_temp = {}
    data = data_split[key]

    # adapt column index to data file

    dict_temp['XSignal1'] = - data[0:cut, 0] + np.mean(data[cut:, 0])
    dict_temp['XTrap1'] = data[0:cut, 6] - np.mean(data[cut:, 6])
    dict_temp['Time'] = data[0:cut, 11]
    dict_temp['XSignal2'] = - data[0:cut, 3] + np.mean(data[cut:, 3])
    dict_temp['DELTA1'] = dict_temp['XSignal1']/S1
    dict_temp['DELTA2'] = dict_temp['XSignal2']/S2
    
    dict_temp['F2'] = k2 * dict_temp['DELTA2'] * 10**12
    dict_temp['delta'] = (distance_init - dict_temp['DELTA1']) * 10**9

    dict_temp['F2_smooth'] = pd.Series(dict_temp['F2']).rolling(window=30, center=True).mean()

    dict[key] = dict_temp

    plt.plot(dict_temp['Time'], dict_temp['F2_smooth'], label=key, marker='.')

plt.xlabel("Time (s)",fontsize=30)
plt.ylabel("Force (pN)",fontsize=30)
plt.xticks(fontsize=30)
plt.yticks(fontsize=30)
plt.title(f"Initial distance = {distance_init} m", fontsize=20)
plt.tight_layout()
plt.show

fig1.savefig(path+nom_figure+'_force-time.png', dpi=250)

### Force-Distance plotting

In [ ]:
# cut indexes
cut1, cut2 = 20480-1, 30720-1

excluded_keys = [7,8,9,10]  # exclude keys if data cannot be used
num_plots = len(dict) - len(excluded_keys)

fig2, axes = plt.subplots(nrows=num_plots, ncols=1, figsize=(10, 2 * num_plots), sharex=True)

all_distance_part1 = []
all_distance_part3 = []
all_force_part1 = []
all_force_part3 = []

i = 0  # Compteur pour savoir sur quel subplot on est

for key in dict:
    if key in excluded_keys:  # Vérifie si la clé est dans la liste d'exclusion
        continue  # Passe à la suivante

    data = dict[key]

    XTrap1_part1, XTrap1_part3 = data['XTrap1'][:cut1], data['XTrap1'][cut2:]
    DELTA1_part1, DELTA1_part3 = data['DELTA1'][:cut1], data['DELTA1'][cut2:]
    DELTA2_part1, DELTA2_part3 = data['DELTA2'][:cut1], data['DELTA2'][cut2:]

    # Calcul du déplacement réel (exemple de formule, adapte selon ton calcul)
    displacement_part1 = XTrap1_part1 - DELTA1_part1 - DELTA2_part1
    displacement_part3 = XTrap1_part3 - DELTA1_part3 - DELTA2_part3

    # Calcul de la distance entre particules
    distance_part1 = (distance_init - displacement_part1) *10**9 -950
    distance_part3 = (distance_init - displacement_part3) *10**9 -950

    # Force correspondante
    force_part1 = data['F2_smooth'][:cut1]
    force_part3 = data['F2_smooth'][cut2:]

    # Stockage des données pour la moyenne
    all_distance_part1.append(np.flip(distance_part1))
    all_distance_part3.append(distance_part3)
    all_force_part1.append(np.flip(force_part1))
    all_force_part3.append(force_part3)
    
    # Tracer sur le subplot correspondant
    ax = axes[i]  # Sélectionne l'axe du subplot actuel
    ax.plot(distance_part1, force_part1, linestyle='dotted', label=f"{key} - Aller")
    ax.plot(distance_part3, force_part3, label=f"{key} - Retour")

    ax.set_ylabel("Force (pN)")
    ax.legend()
    ax.grid()
    #ax.set_title(f"Graph for {key}")

    i += 1  # Passe au subplot suivant

# Ajout d'un label commun pour l'axe X
plt.xlabel("Distance between particles (nm)")
plt.tight_layout()
plt.show()

fig2.savefig(path+nom_figure+'_force-distance.png', dpi=250)



### Mean Force-Distance plotting and saving txt data

In [ ]:
fig3, ax3 = plt.subplots(figsize=(10, 6))

# Checking lists are not empty
if all_distance_part1 and all_distance_part3:
    # Creation of a common distance grid
    common_dist1 = np.linspace(min(np.concatenate(all_distance_part1)), max(np.concatenate(all_distance_part1)), 10000)
    common_dist3 = np.linspace(min(np.concatenate(all_distance_part3)), max(np.concatenate(all_distance_part3)), 10000)

    # Interpolation of forces on the common grid
    interp_forces_part1 = [np.interp(common_dist1, dist, force) for dist, force in zip(all_distance_part1, all_force_part1)]
    interp_forces_part3 = [np.interp(common_dist3, dist, force) for dist, force in zip(all_distance_part3, all_force_part3)]

    # Calculation of mean force
    mean_force_part1 = np.mean(interp_forces_part1, axis=0)
    mean_force_part3 = np.mean(interp_forces_part3, axis=0)

    ax3.plot(common_dist1, mean_force_part1, label="Approche", color="blue")
    ax3.plot(common_dist3, mean_force_part3, label="Rétractation", color="red")

    ax3.set_xlabel("Distance (nm)", fontsize=14)
    ax3.set_ylabel("Force (pN)", fontsize=14)
    ax3.legend(fontsize=14)
    ax3.grid()
    #plt.title("Mean of approaches and retractaions")

    plt.tight_layout()
print(f"Nombre de courbes stockées : {len(all_distance_part1)} allers, {len(all_distance_part3)} retours")
for i, (d, f) in enumerate(zip(all_distance_part1, all_force_part1)):
    print(f"Approach {i+1}: {len(d)} points")
for i, (d, f) in enumerate(zip(all_distance_part3, all_force_part3)):
    print(f"Retraction {i+1}: {len(d)} points")

    plt.show()

    fig3.savefig(path + nom_figure + '_mean_force-distance.png', dpi=250)


else:
    print("Error : data lists are empty.")

    # ---- Export en fichier TXT ----
output_file = path2 + nom_figure + '_mean_force_distance.txt'
with open(output_file, 'w') as f:
    f.write("Distance Approach (nm), Force Approach (pN), Distance Retraction (nm), Force Retraction (pN)\n")
    for d1, f1, d3, f3 in zip(common_dist1, mean_force_part1, common_dist3, mean_force_part3):
        f.write(f"{d1:.6f}, {f1:.6f}, {d3:.6f}, {f3:.6f}\n")

print(f"Exported file : {output_file}")